# Neo4j docs examples

This is a modification of https://github.com/neo4j/neo4j-graphrag-python. This example uses the ollama and embedder open source.

In [25]:
%pip install -r requirements.txt

  Using cached html2text-2025.4.15-py3-none-any.whl.metadata (4.1 kB)
Using cached html2text-2025.4.15-py3-none-any.whl (34 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
from dotenv import load_dotenv
import os

if load_dotenv():
    print("Success: .env file found with some environment variables")
else:
    print("Caution: No environment variables found. Please create .env file in the root directory or add environment variables in the .env file")

Success: .env file found with some environment variables


## 1.Knowledge Graph Construction

Firs you have to set the neo4j service listening on 7687 port. I use the neo4j docker container

In [2]:
import asyncio

from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import OpenAILLM

In [3]:
NEO4J_URI = os.getenv('DATABASE_URL')
NEO4J_USERNAME = os.getenv('NEO4J_USER')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
OLLAMA_URL= os.getenv('OLLAMA_URL')

In [44]:
# Connect to the Neo4j database
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))


Embedder and llm model

In [14]:
from neo4j_graphrag.llm import OllamaLLM
from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings

llm = OllamaLLM(
    model_name="qwen2.5:14b"
    )

embedder = SentenceTransformerEmbeddings(
    model="sentence-transformers/all-mpnet-base-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 15158.12it/s]


### 1.1.Dune Example

In [ ]:
# List the entities and relations the LLM should look for in the text
node_types = ["Person", "House", "Planet"]
relationship_types = ["PARENT_OF", "HEIR_OF", "RULES"]
patterns = [
    ("Person", "PARENT_OF", "Person"),
    ("Person", "HEIR_OF", "House"),
    ("House", "RULES", "Planet"),
]

# Instantiate the SimpleKGPipeline
kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    schema={
        "node_types": node_types,
        "relationship_types": relationship_types,
        "patterns": patterns,
    },
    on_error="IGNORE",
    from_file=False,
)

In [47]:

# Run the pipeline on a piece of text
text = (
    "The son of Duke Leto Atreides and the Lady Jessica, Paul is the heir of House "
    "Atreides, an aristocratic family that rules the planet Caladan."
)
await kg_builder.run_async(text=text)
driver.close()

This piece of code is for remove the kwnowledege graph

In [5]:
driver.execute_query("MATCH (n) DETACH DELETE n")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7b4606e84aa0>, keys=[])

### 1.2. 2024 UEFA Euro Championship example

In [9]:
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer

url="https://en.wikipedia.org/wiki/UEFA_Euro_2024"

loader = AsyncHtmlLoader (url)

html_data = loader.load()

#Assign the Html2TextTransformer function
html2text = Html2TextTransformer()

#Call transform_documents
html_data_transformed = html2text.transform_documents(html_data)
text=html_data_transformed[0].page_content

USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.15it/s]


In [13]:
text

'Jump to content\n\nMain menu\n\nMain menu\n\nmove to sidebar hide\n\nNavigation\n\n  * Main page\n  * Contents\n  * Current events\n  * Random article\n  * About Wikipedia\n  * Contact us\n\nContribute\n\n  * Help\n  * Learn to edit\n  * Community portal\n  * Recent changes\n  * Upload file\n  * Special pages\n\nSearch\n\nSearch\n\nAppearance\n\n  * Donate\n  * Create account\n  * Log in\n\nPersonal tools\n\n  * Donate\n  * Create account\n  * Log in\n\n## Contents\n\nmove to sidebar hide\n\n  * (Top)\n\n  * 1 Host selection\n\n  * 2 Venues\n\nToggle Venues subsection\n\n    * 2.1 Team base camps\n\n    * 2.2 Ticketing\n\n  * 3 Qualification\n\nToggle Qualification subsection\n\n    * 3.1 Qualified teams\n\n      * 3.1.1 Disqualification of Russia\n\n  * 4 Final draw\n\nToggle Final draw subsection\n\n    * 4.1 Seeding\n\n    * 4.2 Draw\n\n  * 5 Squads\n\n  * 6 Match officials\n\n  * 7 Group stage\n\nToggle Group stage subsection\n\n    * 7.1 Tiebreakers\n\n    * 7.2 Group A\n\n    * 

In [39]:
node_types = [
    "Tournament",
    "Team",
    "Player",
    "Coach",
    "Match",
    "Stadium",
    "City",
    "Country",
    "Stage"
]

relationship_types = [
    "PARTICIPATES_IN",
    "PLAYS_FOR",
    "COACHES",
    "HOSTED_IN",
    "LOCATED_IN",
    "PLAYED_AT",
    "PLAYED_IN",
    "WON_BY",
    "ADVANCED_TO",
    "COMPETED_AGAINST"
]

patterns = [

    # Teams in tournament
    ("Team", "PARTICIPATES_IN", "Tournament"),

    # Players and coaches
    ("Player", "PLAYS_FOR", "Team"),
    ("Coach", "COACHES", "Team"),

    # Stadium location
    ("Stadium", "LOCATED_IN", "City"),
    ("City", "LOCATED_IN", "Country"),

    # Match relations
    ("Match", "PLAYED_AT", "Stadium"),
    ("Match", "PLAYED_IN", "Stage"),

    # Teams competing
    ("Team", "COMPETED_AGAINST", "Team"),

    # Tournament progression
    ("Team", "ADVANCED_TO", "Stage"),

    # Winner
    ("Tournament", "WON_BY", "Team"),
]

# Instantiate the SimpleKGPipeline
kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    schema={
        "node_types": node_types,
        "relationship_types": relationship_types,
        "patterns": patterns,
    },
    from_file=False,
    on_error="IGNORE",
)

In [16]:
await kg_builder.run_async(text=text)

SchemaExtractionError: LLM response is not valid JSON.

In [34]:
from bs4 import BeautifulSoup

def clean_html(html):

    soup = BeautifulSoup(html, "html.parser")

    for tag in soup([
        "script",
        "style",
        "nav",
        "footer",
        "header",
        "aside"
    ]):
        tag.decompose()

    return soup.get_text(separator=" ", strip=True)

def chunk_text(text, chunk_size=800, overlap=100):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks



In [40]:
chunks = chunk_text(text)

In [41]:
for chunk in chunks:
    await kg_builder.run_async(text=chunk)

LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0
LLM response has improper format for chunk_index=0


In [27]:
len(chunks)

179

In [37]:
driver.execute_query("MATCH (n) DETACH DELETE n")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7b42f8b24650>, keys=[])

## 2. Vector index

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_vector_index

INDEX_NAME = "vector-index-opensource"

# Create the index
create_vector_index(
    driver,
    INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=768,
    similarity_fn="cosine",
)
#driver.close()

## 3. Retriever

In [46]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.retrievers import VectorRetriever

# Initialize the retriever
retriever = VectorRetriever(driver, INDEX_NAME, embedder)
# Instantiate the RAG pipeline
rag = GraphRAG(retriever=retriever, llm=llm)

In [50]:
# Query the graph
query_text = "Which teams played the final match in the 2024 UEFA European Championship?"
response = rag.search(query_text=query_text, retriever_config={"top_k": 5})
print(response.answer)
#driver.close()

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node { .*, `embedding`: null } AS node, labels(node) AS nodeLabels, elementId(node) AS elementId, elementId(node) AS id, score'


The provided context does not specify which teams played in the final match of the 2024 UEFA European Championship. The information available only mentions that the final was held at the Olympiastadion in Berlin but does not provide details about the competing teams.
